<div align="center" style="background:linear-gradient(135deg,#0A0A0A 0%,#1A1A1A 50%,#0D0D0D 100%);border:1.5px solid #C9A84C;border-radius:18px;padding:34px 24px 30px;margin:10px 0 20px;box-shadow:0 0 36px rgba(201,168,76,0.18),0 8px 32px rgba(0,0,0,0.6);font-family:Arial,sans-serif;text-align:center;"><div style="width:65%;height:3px;margin:0 auto 22px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:4px;"></div><p align="center" style="color:#C9A84C;letter-spacing:5px;font-size:11px;font-weight:700;text-transform:uppercase;margin:0 0 10px;text-align:center;">&#10022; TUSHKOSH 2.0 PRESENTS &#10022;</p><h1 align="center" style="color:#FFFFFF;font-size:2.05em;margin:0 0 8px;text-shadow:0 2px 12px rgba(201,168,76,0.4);letter-spacing:1px;text-align:center;">Fish Audio S2 1 Pro</h1><div style="width:80px;height:2px;margin:12px auto;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:2px;"></div><p align="center" style="color:#C9A84C;font-size:13px;font-weight:700;letter-spacing:3px;margin:0 0 22px;text-transform:uppercase;text-align:center;">Learn &bull; Build &bull; Create</p><div align="center" style="text-align:center;margin:0 0 18px;"><a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" style="display:inline-block;background:linear-gradient(135deg,#C9A84C 0%,#FFD700 50%,#C9A84C 100%);color:#000000;font-weight:800;font-size:13px;padding:12px 30px;border-radius:50px;text-decoration:none;letter-spacing:1px;box-shadow:0 4px 20px rgba(201,168,76,0.5);">&#9654; Subscribe to TushKosh 2.0</a></div><div align="center" style="text-align:center;margin-top:6px;"><img src="https://img.shields.io/badge/TushKosh%202.0-AI%20Tutorials-C9A84C?style=flat-square&labelColor=0A0A0A" style="margin:3px;vertical-align:middle;" /><img src="https://img.shields.io/badge/Original%20by-AIQUEST%20Academy-555555?style=flat-square&labelColor=1A1A1A" style="margin:3px;vertical-align:middle;" /></div><div style="width:65%;height:2px;margin:22px auto 0;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:4px;"></div></div>

In [ ]:
#@title 🚀 Step 1: Launch Fish Audio S2.1 Pro Studio (Complete All-in-One)
# Installs dependencies, configures environment, and launches the modern Gradio 5+ UI.
# Runs on Google Colab standard free CPU runtime (no GPU required).
import os
import sys
import time
import tempfile
import subprocess
import requests

print("=" * 68)
print("🐟 Fish Audio S2.1 Pro - Modern Interactive Studio (100% Free Tier)")
print("Colab Edition (CPU Friendly) - TushKosh 2.0")
print("=" * 68)

# 1. Check and install modern dependencies
print("\n[1/3] Ensuring modern Gradio 5+ and Fish Audio SDK are installed...")
packages = [
    "fish-audio-sdk>=1.0.0",
    "gradio>=5.0.0",
    "soundfile",
    "requests"
]
try:
    import fishaudio
    import gradio as gr
    from fishaudio import FishAudio
    from fishaudio.types import ReferenceAudio
    gr_major = int(gr.__version__.split(".")[0])
    if gr_major < 5:
        raise ImportError("Upgrading to modern Gradio 5+")
    print(f"✅ Modern Gradio {gr.__version__} and Fish Audio SDK ready!")
except Exception:
    print("📦 Installing/Upgrading packages to modern standards...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages, check=True)
    import gradio as gr
    from fishaudio import FishAudio
    from fishaudio.types import ReferenceAudio
    print(f"✅ Installed Gradio {gr.__version__} successfully!")

# 2. Check API Key
print("\n[2/3] Checking API Key...")
api_key_loaded = False
try:
    from google.colab import userdata
    colab_key = userdata.get('FISH_API_KEY')
    if colab_key and colab_key.strip():
        os.environ['FISH_API_KEY'] = colab_key.strip()
        api_key_loaded = True
        print("✅ FISH_API_KEY securely loaded from Google Colab Secrets!")
except Exception:
    pass

if not api_key_loaded:
    if os.environ.get("FISH_API_KEY", "").strip():
        print("✅ FISH_API_KEY detected in environment variables!")
    else:
        print("ℹ️ FISH_API_KEY not configured in Secrets. You can enter it directly in the UI.")
        print("👉 Get your API key here: https://docs.fish.audio/developer-guide/getting-started/api-key")

# 3. Studio Backend Engine
print("\n[3/3] Launching Modern Studio Interface...")

def get_client(api_key: str = "") -> FishAudio:
    clean_key = (api_key or "").strip() or os.environ.get("FISH_API_KEY", "").strip()
    if not clean_key:
        raise gr.Error("Missing API Key! Please enter your Fish Audio API Key in the field above or set FISH_API_KEY in Colab Secrets.")
    return FishAudio(api_key=clean_key)

def get_user_voice_choices(api_key: str = "", filter_mode: str = "All Voices"):
    clean_key = (api_key or "").strip() or os.environ.get("FISH_API_KEY", "").strip()
    choices = [("⭐ Default Base Narrator (Fish Audio)", "")]

    # 1. Custom account voices
    if clean_key and filter_mode in ["All Voices", "My Voices Only"]:
        try:
            client = FishAudio(api_key=clean_key)
            my_res = client.voices.list(page_size=50, self_only=True)
            my_items = getattr(my_res, "items", [])
            for v in my_items:
                choices.append((f"👤 [My Voice] {v.title} ({v.id[:8]}...)", v.id))
        except Exception as e:
            print(f"Notice: Could not load custom voices ({e})")

    # 2. System and featured platform voices
    if filter_mode in ["All Voices", "System Voices Only"]:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            if clean_key:
                headers["Authorization"] = f"Bearer {clean_key}"
            resp = requests.get("https://api.fish.audio/model?sort_by=task_count&page_size=40", headers=headers, timeout=8)
            if resp.status_code == 200:
                data = resp.json()
                for item in data.get("items", []):
                    v_id = item.get("_id")
                    v_title = item.get("title", "Untitled")
                    tags = item.get("tags", [])
                    tag_str = f" [{tags[0]}]" if tags else ""
                    choices.append((f"🌐 [System Voice] {v_title}{tag_str} ({v_id[:8]}...)", v_id))
        except Exception as e:
            print(f"Notice: Could not load system voices ({e})")

    return gr.Dropdown(choices=choices, value="")

def generate_expressive_tts(api_key, model_choice, text_input, voice_dropdown_val, manual_ref_id, audio_format):
    if not text_input or not text_input.strip():
        raise gr.Error("Please enter some text to synthesize.")

    start_time = time.time()
    client = get_client(api_key)

    chosen_ref_id = (manual_ref_id or "").strip() or (voice_dropdown_val or "").strip()
    ref_id = chosen_ref_id if chosen_ref_id else None
    kwargs = {
        "text": text_input.strip(),
        "model": model_choice or "s2.1-pro-free",
    }
    if ref_id:
        kwargs["reference_id"] = ref_id
    if audio_format:
        kwargs["format"] = audio_format

    try:
        audio_bytes = client.tts.convert(**kwargs)
    except Exception as e:
        raise gr.Error(f"TTS Generation failed: {str(e)}")

    duration = time.time() - start_time
    ext = audio_format if audio_format in ["mp3", "wav", "opus"] else "mp3"
    with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False) as f:
        f.write(audio_bytes)
        out_file = f.name

    stats = f"⚡ **Generated in {duration:.2f}s** | Model: `{model_choice}` | Format: `{ext.upper()}`"
    return out_file, stats

def generate_voice_clone(api_key, model_choice, ref_audio, target_text, audio_format):
    if not ref_audio:
        raise gr.Error("Please upload or record reference audio for voice cloning.")
    if not target_text or not target_text.strip():
        raise gr.Error("Please enter the target text for the cloned voice to speak.")

    start_time = time.time()
    client = get_client(api_key)

    with open(ref_audio, "rb") as f:
        audio_bytes_ref = f.read()

    ref_obj = ReferenceAudio(
        audio=audio_bytes_ref,
        text=""
    )

    kwargs = {
        "text": target_text.strip(),
        "references": [ref_obj],
        "model": model_choice or "s2.1-pro-free",
    }
    if audio_format:
        kwargs["format"] = audio_format

    try:
        audio_bytes = client.tts.convert(**kwargs)
    except Exception as e:
        raise gr.Error(f"Voice Cloning failed: {str(e)}")

    duration = time.time() - start_time
    ext = audio_format if audio_format in ["mp3", "wav", "opus"] else "mp3"
    with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False) as f:
        f.write(audio_bytes)
        out_file = f.name

    stats = f"⚡ **Cloned & Synthesized in {duration:.2f}s** | Model: `{model_choice}` | Format: `{ext.upper()}`"
    return out_file, stats

def create_persistent_voice(api_key, sample_audio, voice_title, voice_desc, visibility):
    if not sample_audio:
        raise gr.Error("Please upload a clean audio sample (10-30s) to create your voice model.")
    if not voice_title or not voice_title.strip():
        raise gr.Error("Please provide a title for your voice model.")

    client = get_client(api_key)
    with open(sample_audio, "rb") as f:
        audio_data = f.read()

    try:
        voice = client.voices.create(
            title=voice_title.strip(),
            voices=[audio_data],
            description=(voice_desc or "").strip(),
            visibility=visibility
        )
        return (
            f"🎉 **Custom Voice Model Created!**\n\n"
            f"- **Voice ID:** `{voice.id}`\n"
            f"- **Title:** {voice.title}\n"
            f"- **State:** {getattr(voice, 'state', 'created')}\n\n"
            f"💡 *Copy this Voice ID into the **Reference Voice ID** field in Tab 1 to use it anytime.*"
        )
    except Exception as e:
        raise gr.Error(f"Failed to create voice model: {str(e)}")

def list_user_voices(api_key, self_only):
    client = get_client(api_key)
    try:
        res = client.voices.list(page_size=25, self_only=self_only)
        items = getattr(res, "items", [])
        if not items:
            return "No voice models found."

        table = ["| Voice Title | Reference Voice ID | Tags | State |", "| :--- | :--- | :--- | :--- |"]
        for v in items:
            tags = ", ".join(getattr(v, 'tags', [])) if getattr(v, 'tags', []) else "custom"
            state = getattr(v, 'state', 'ready')
            table.append(f"| **{v.title}** | `{v.id}` | {tags} | {state} |")
        return "\n".join(table)
    except Exception as e:
        raise gr.Error(f"Failed to retrieve voices: {str(e)}")

def insert_emotion_tag(current_text, tag):
    curr = current_text or ""
    if curr and not curr.endswith(" "):
        return curr + " " + tag + " "
    return curr + tag + " "

# ---------------------------------------------------------
# Modern Styling & Custom Theme
# ---------------------------------------------------------
modern_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="purple",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Plus Jakarta Sans"), gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"]
).set(
    body_background_fill="#f8fafc",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#e2e8f0",
    block_radius="16px",
    button_primary_background_fill="linear-gradient(135deg, #4f46e5 0%, #7c3aed 100%)",
    button_primary_background_fill_hover="linear-gradient(135deg, #4338ca 0%, #6d28d9 100%)",
    button_primary_text_color="white",
    input_background_fill="#f8fafc",
    input_border_color="#cbd5e1",
    input_radius="12px"
)

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');
* { font-family: 'Plus Jakarta Sans', sans-serif !important; }
.gradio-container { max-width: 1120px !important; margin: auto !important; }

/* Modern Hero Banner */
.brand-hero {
    text-align: center;
    background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 50%, #9333ea 100%);
    padding: 36px 24px;
    border-radius: 20px;
    margin-bottom: 24px;
    box-shadow: 0 20px 40px rgba(79, 70, 229, 0.22);
    position: relative;
    overflow: hidden;
}
.brand-hero::after {
    content: '';
    position: absolute;
    top: -50%;
    left: -50%;
    width: 200%;
    height: 200%;
    background: radial-gradient(circle, rgba(255,255,255,0.12) 0%, transparent 60%);
    pointer-events: none;
}
.hero-badge {
    display: inline-block;
    padding: 5px 14px;
    background: rgba(255,255,255,0.18);
    border: 1px solid rgba(255,255,255,0.28);
    border-radius: 9999px;
    color: white;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    margin-bottom: 12px;
}
.hero-title {
    color: white !important;
    font-size: 2.5em !important;
    font-weight: 800 !important;
    margin: 0 0 8px 0 !important;
    letter-spacing: -0.02em !important;
}
.hero-subtitle {
    color: rgba(255, 255, 255, 0.92) !important;
    font-size: 1.1em !important;
    margin-bottom: 6px !important;
}
.hero-desc {
    color: rgba(255, 255, 255, 0.78) !important;
    font-size: 0.95em !important;
    margin: 0 !important;
    text-align: center !important;
}
.hero-socials {
    display: flex;
    justify-content: center;
    gap: 12px;
    flex-wrap: wrap;
    margin-top: 18px;
}
.hero-social-btn {
    padding: 9px 22px;
    border-radius: 10px;
    font-weight: 700;
    font-size: 13px;
    text-decoration: none;
    display: inline-flex;
    align-items: center;
    gap: 8px;
    color: white !important;
    transition: all 0.25s ease;
    box-shadow: 0 4px 14px rgba(0,0,0,0.25);
}
.hero-social-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 8px 22px rgba(0,0,0,0.35);
}
.btn-youtube { background: linear-gradient(135deg, #ef4444 0%, #dc2626 100%); }
.btn-x { background: linear-gradient(135deg, #000000 0%, #333333 100%); }

/* Modern Interactive Emotion Chips */
.tag-chip-btn {
    border: 1px solid #e0e7ff !important;
    background: #eef2ff !important;
    color: #4338ca !important;
    border-radius: 9999px !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    padding: 5px 12px !important;
    transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
    box-shadow: 0 1px 2px rgba(0,0,0,0.03) !important;
}
.tag-chip-btn:hover {
    background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 100%) !important;
    color: white !important;
    border-color: transparent !important;
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 16px rgba(79, 70, 229, 0.3) !important;
}
.tag-phrase-btn {
    border-radius: 10px !important;
    white-space: normal !important;
    text-align: left !important;
    line-height: 1.4 !important;
    font-size: 11.5px !important;
}

/* Action Buttons per Branding */
.action-gen-btn {
    background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    border: none !important;
    box-shadow: 0 6px 20px rgba(79, 70, 229, 0.3) !important;
    transition: all 0.2s ease !important;
}
.action-gen-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 10px 25px rgba(79, 70, 229, 0.45) !important;
}
.action-stop-btn {
    background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    border: none !important;
    box-shadow: 0 4px 14px rgba(239, 68, 68, 0.25) !important;
}
.action-stop-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 20px rgba(239, 68, 68, 0.35) !important;
}
.action-clear-btn {
    background: linear-gradient(135deg, #64748b 0%, #334155 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    border: none !important;
    box-shadow: 0 4px 14px rgba(100, 116, 139, 0.2) !important;
}
.action-clear-btn:hover {
    transform: translateY(-2px) !important;
}

/* Pro Tips Card */
.pro-tips-card {
    background: #f1f5f9;
    border: 1px solid #cbd5e1;
    border-radius: 12px;
    padding: 14px;
    margin-top: 12px;
}

/* Footer */
.modern-footer {
    text-align: center;
    padding: 28px 16px;
    margin-top: 36px;
    border-top: 1px solid #e2e8f0;
    color: #64748b;
}


/* TushKosh 2.0 - Premium Black & Gold Theme */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');
* { font-family: 'Inter', sans-serif !important; }
body, .gradio-container { background: #0A0A0A !important; color: #E8E8E8 !important; }
.gradio-container { max-width: 1050px !important; margin: auto !important; }
h1, h2, h3 { color: #C9A84C !important; }

/* Header & Hero blocks */
.brand-header, .hero-header {
  text-align: center !important;
  background: linear-gradient(135deg, #0A0A0A 0%, #1A1A1A 50%, #0D0D0D 100%) !important;
  border: 1.5px solid #C9A84C !important;
  border-radius: 18px !important;
  padding: 28px 24px !important;
  margin-bottom: 20px !important;
  box-shadow: 0 0 36px rgba(201,168,76,0.18), 0 8px 32px rgba(0,0,0,0.6) !important;
}
.brand-title, .hero-title {
  color: #FFFFFF !important;
  text-align: center !important;
  font-weight: 800 !important;
}
.brand-subtitle, .hero-subtitle {
  color: #C9A84C !important;
  text-align: center !important;
  font-weight: 600 !important;
}
.brand-footer, .footer {
  text-align: center !important;
  color: #888888 !important;
  background: #0D0D0D !important;
  border: 1px solid #C9A84C !important;
  border-radius: 14px !important;
  padding: 16px !important;
  margin-top: 20px !important;
}
.btn-row {
  display: flex !important;
  justify-content: center !important;
  align-items: center !important;
  gap: 10px !important;
  flex-wrap: wrap !important;
}
.social-btn, .yt-btn, .hero-social-btn {
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  background: linear-gradient(135deg, #C9A84C 0%, #FFD700 50%, #C9A84C 100%) !important;
  color: #000000 !important;
  font-weight: 800 !important;
  border-radius: 50px !important;
  text-decoration: none !important;
  box-shadow: 0 4px 18px rgba(201,168,76,0.45) !important;
}
.x-btn, .sup-btn { display: none !important; }

button.primary, .gr-button-primary {
  background: linear-gradient(135deg, #C9A84C 0%, #FFD700 50%, #C9A84C 100%) !important;
  color: #000000 !important; font-weight: 800 !important;
  border-radius: 50px !important; border: none !important;
  box-shadow: 0 4px 20px rgba(201,168,76,0.4) !important;
}
input, textarea, select, .gr-input, .gr-textbox {
  background: #111111 !important; border: 1px solid #333333 !important;
  color: #E8E8E8 !important; border-radius: 10px !important;
}
input:focus, textarea:focus {
  border-color: #C9A84C !important;
  box-shadow: 0 0 0 2px rgba(201,168,76,0.2) !important;
}
label, .gr-form > label { color: #C9A84C !important; font-weight: 600 !important; }
.gr-panel, .gr-block, .gr-box, .gr-form {
  background: #111111 !important; border: 1px solid #222222 !important;
  border-radius: 14px !important;
}
input[type=range]::-webkit-slider-thumb { background: #C9A84C !important; }
input[type=range]::-webkit-slider-runnable-track { background: #333 !important; }
"""

PRESET_SCRIPTS = {
    "Confrontation & Betrayal (Throat Clear + Anger)": (
        "[clears throat] [angry] How could you?! [angry] [emphasis] "
        "After everything we sacrificed together, you sold us out without a second thought!"
    ),
    "Ticklish Play (Laughing + Excited)": (
        "[laughing] [excited] Hey! Stop tickling me! [laughing] [breathless] "
        "I literally can't breathe right now, please stop!"
    ),
    "Exhausted Marathon (Panting + Tired)": (
        "[panting] [tired] I've been running for miles... [breathless] [sighing] "
        "I don't think my legs can take another single step."
    ),
    "Helpless Resignation (Sighing + Emphasis)": (
        "[sighing] [emphasis] I just don't know anymore. [sighs] [sad] "
        "We threw every dollar and every hour into this, and it still fell apart."
    ),
    "Master Surgeon (Open-Domain Measured Tone)": (
        "[the calm, measured tone of someone who has done this a thousand times] "
        "Clamp the femoral artery and stabilize the patient's pressure. We have plenty of time. Keep your hands steady."
    ),
    "Graveyard Shift (Open-Domain Dead Tired)": (
        "[dead tired, end of a very long shift] "
        "That's the final inspection report filed for tonight... [sighing] Can someone please tell me my relief has arrived?"
    ),
    "Holding Back Tears (Open-Domain Masking Emotion)": (
        "[voice rough from crying, trying to sound normal] "
        "No, honestly, I'm completely fine... [sniffling] [voice breaking] Just give me a second to grab a glass of water."
    ),
    "Whispering Suspense & Fear": (
        "[whispering] [fearful] Shh! Don't make a sound... [gasping] I think someone is right outside that door. "
        "[whispering] [softly] Whatever you do, stay down and don't breathe."
    ),
    "Triumphant Excitement & Shouting": (
        "[excited] [shouting] Yes! We actually did it! [excited] [breathless] "
        "The benchmark scores just came in and we officially crushed the world record! [laughing] [fast paced] Let's go!"
    ),
    "Biting Sarcasm & Scornful Wit": (
        "[sarcastic] [emphasis] Oh brilliant, another emergency meeting at five PM on a Friday. "
        "[sarcastic] [softly] Truly, just what my weekend was missing. [sighs] [slow paced] I am thrilled beyond words."
    )
}

# ---------------------------------------------------------
# UI Construction with Gradio 5+ Blocks
# ---------------------------------------------------------
with gr.Blocks(title="Fish Audio S2.1 Pro Studio - TushKosh 2.0", theme=modern_theme, css=custom_css) as demo:
    # Modern Hero Header
    gr.HTML("""
    <div class="brand-hero">
      <div class="hero-badge">100% Free Tier • Cloud API Edition</div>
      <h1 class="hero-title">🐟 Fish Audio S2.1 Pro Studio</h1>
      <h3 class="hero-subtitle">Colab Edition (CPU Friendly & 100% Free) - Created by <strong>TushKosh 2.0</strong></h3>
      <p class="hero-desc">Dual-Autoregressive Voice Synthesis • Zero-Shot Voice Cloning • Paralinguistic Emotion Control</p>
      <div class="hero-socials">
        <a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" class="hero-social-btn btn-youtube">
          <svg style="width:16px;height:16px;fill:currentColor" viewBox="0 0 24 24"><path d="M23.498 6.186a3.016 3.016 0 0 0-2.122-2.136C19.505 3.545 12 3.545 12 3.545s-7.505 0-9.377.505A3.017 3.017 0 0 0 .502 6.186C0 8.07 0 12 0 12s0 3.93.502 5.814a3.016 3.016 0 0 0 2.122 2.136c1.871.505 9.376.505 9.376.505s7.505 0 9.377-.505a3.015 3.015 0 0 0 2.122-2.136C24 15.93 24 12 24 12s0-3.93-.502-5.814zM9.545 15.568V8.432L15.818 12l-6.273 3.568z"/></svg>
          Subscribe on YouTube
        </a>
        <a href="https://www.youtube.com/@TushKosh2.0" target="_blank" class="hero-social-btn btn-x">
          <svg style="width:15px;height:15px;fill:currentColor" viewBox="0 0 24 24"><path d="M18.244 2.25h3.308l-7.227 8.26 8.502 11.24H16.17l-5.214-6.817L4.99 21.75H1.68l7.73-8.835L1.254 2.25H8.08l4.713 6.231zm-1.161 17.52h1.833L7.084 4.126H5.117z"/></svg>
          Follow on X
        </a>
      </div>
    </div>
    """)

    # Global API Key Configuration
    with gr.Accordion("🔑 API Key Configuration (Fish Audio)", open=True):
        default_env_key = os.environ.get("FISH_API_KEY", "")
        with gr.Row():
            api_key_input = gr.Textbox(
                label="Fish Audio API Key",
                placeholder="Enter your API Key here (or leave blank if loaded from Colab Secrets)",
                value=default_env_key,
                type="password",
                scale=4
            )
            api_status_badge = gr.Markdown(
                value="✅ **Key Loaded**" if default_env_key else "⚠️ **Key Required**",
                scale=1
            )
        gr.Markdown(
            "💡 *Tip: Get your API key here: [https://docs.fish.audio/developer-guide/getting-started/api-key](https://docs.fish.audio/developer-guide/getting-started/api-key). In Google Colab, add it to **Secrets** as `FISH_API_KEY` for automatic zero-click loading, or paste it directly in the box above.*"
        )

    # Main Tabs
    with gr.Tabs():
        # -----------------------------------------------------
        # TAB 1: Expressive TTS & Paralinguistic Tags
        # -----------------------------------------------------
        with gr.Tab("🎭 Expressive TTS & Emotion Tags"):
            with gr.Row():
                with gr.Column(scale=6):
                    with gr.Row():
                        model_select_t1 = gr.Dropdown(
                            label="Model Selection",
                            choices=["s2.1-pro-free", "s2.1-pro"],
                            value="s2.1-pro-free",
                            info="s2.1-pro-free: 100% Free Developer Tier (Default). s2.1-pro: Production tier.",
                            scale=3
                        )
                        format_t1 = gr.Radio(
                            label="Format",
                            choices=["mp3", "wav", "opus"],
                            value="mp3",
                            scale=2
                        )

                    preset_dropdown = gr.Dropdown(
                        label="Sample Expressive Prompts (Multi-Tag & Open-Domain Presets)",
                        choices=list(PRESET_SCRIPTS.keys()),
                        value="Confrontation & Betrayal (Throat Clear + Anger)",
                        info="Select a curated preset to test physical+emotion pairings or open-domain acting directives."
                    )

                    text_input_t1 = gr.Textbox(
                        label="Input Text (Pair physical+emotion tags or use open-domain phrases like [dead tired, end of shift])",
                        placeholder="e.g.: [clears throat] [angry] How could you?! ... [the calm, measured tone of someone who has done this a thousand times]",
                        value=PRESET_SCRIPTS["Confrontation & Betrayal (Throat Clear + Anger)"],
                        lines=5
                    )

                    # Modern Interactive Tag Chips
                    with gr.Accordion("🏷️ Paralinguistic & Emotion Tag Chips (Click to Insert at Cursor)", open=True):
                        gr.Markdown("**🎭 Primary Emotions:**")
                        with gr.Row():
                            t_happy = gr.Button("😊 [happy]", elem_classes=["tag-chip-btn"])
                            t_sad = gr.Button("😢 [sad]", elem_classes=["tag-chip-btn"])
                            t_angry = gr.Button("😠 [angry]", elem_classes=["tag-chip-btn"])
                            t_excited = gr.Button("🤩 [excited]", elem_classes=["tag-chip-btn"])
                            t_nervous = gr.Button("😰 [nervous]", elem_classes=["tag-chip-btn"])
                            t_calm = gr.Button("😌 [calm]", elem_classes=["tag-chip-btn"])
                            t_sarcastic = gr.Button("😏 [sarcastic]", elem_classes=["tag-chip-btn"])
                            t_fearful = gr.Button("😨 [fearful]", elem_classes=["tag-chip-btn"])
                            t_tired = gr.Button("🥱 [tired]", elem_classes=["tag-chip-btn"])

                        gr.Markdown("**🌬️ Physical Actions & Breath:**")
                        with gr.Row():
                            t_clearthroat = gr.Button("🗣️ [clears throat]", elem_classes=["tag-chip-btn"])
                            t_sighing = gr.Button("😮‍💨 [sighing]", elem_classes=["tag-chip-btn"])
                            t_whisper = gr.Button("🤫 [whispering]", elem_classes=["tag-chip-btn"])
                            t_laugh = gr.Button("😂 [laughing]", elem_classes=["tag-chip-btn"])
                            t_sighs = gr.Button("💨 [sighs]", elem_classes=["tag-chip-btn"])
                            t_giggles = gr.Button("🤭 [giggles]", elem_classes=["tag-chip-btn"])
                            t_crying = gr.Button("😭 [crying]", elem_classes=["tag-chip-btn"])
                            t_gasping = gr.Button("😱 [gasping]", elem_classes=["tag-chip-btn"])
                            t_sniffling = gr.Button("🤧 [sniffling]", elem_classes=["tag-chip-btn"])
                            t_panting = gr.Button("😮 [panting]", elem_classes=["tag-chip-btn"])

                        gr.Markdown("**🎙️ Vocal Delivery & Style:**")
                        with gr.Row():
                            t_softly = gr.Button("🕊️ [softly]", elem_classes=["tag-chip-btn"])
                            t_vbreak = gr.Button("💔 [voice breaking]", elem_classes=["tag-chip-btn"])
                            t_emphasis = gr.Button("⚡ [emphasis]", elem_classes=["tag-chip-btn"])
                            t_shouting = gr.Button("📢 [shouting]", elem_classes=["tag-chip-btn"])
                            t_breathless = gr.Button("🏃 [breathless]", elem_classes=["tag-chip-btn"])
                            t_fast = gr.Button("⏩ [fast paced]", elem_classes=["tag-chip-btn"])
                            t_slow = gr.Button("🐢 [slow paced]", elem_classes=["tag-chip-btn"])

                        gr.Markdown("**🎬 Open-Domain Acting Directives (Click to Insert):**")
                        with gr.Row():
                            t_measured = gr.Button("🩺 [the calm, measured tone of someone who has done this a thousand times]", elem_classes=["tag-chip-btn", "tag-phrase-btn"])
                            t_deadtired = gr.Button("😴 [dead tired, end of a very long shift]", elem_classes=["tag-chip-btn", "tag-phrase-btn"])
                            t_cryingmask = gr.Button("💧 [voice rough from crying, trying to sound normal]", elem_classes=["tag-chip-btn", "tag-phrase-btn"])

                    with gr.Row():
                        voice_filter_t1 = gr.Radio(
                            label="Voice Catalog Filter",
                            choices=["All Voices", "System Voices Only", "My Voices Only"],
                            value="All Voices",
                            scale=3
                        )
                        refresh_voices_btn = gr.Button("🔄 Refresh Voices", scale=1, variant="secondary")

                    voice_dropdown_t1 = gr.Dropdown(
                        label="Select Voice (Default Base, System & Custom Voices)",
                        choices=[("⭐ Default Base Narrator (Fish Audio)", "")],
                        value="",
                        info="Choose between the default base narrator, official/popular system voices, or your custom voices.",
                    )

                    with gr.Accordion("⚙️ Advanced: Manual Voice ID Override", open=False):
                        manual_ref_id_t1 = gr.Textbox(
                            label="Custom / Public Voice ID (Optional)",
                            placeholder="Leave empty to use the selected voice above, or paste an ID to override",
                            value=""
                        )

                    with gr.Row():
                        gen_btn_t1 = gr.Button("🎬 Generate Audio", variant="primary", size="lg", elem_classes=["action-gen-btn"])
                        stop_btn_t1 = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_classes=["action-stop-btn"])
                        clear_btn_t1 = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_classes=["action-clear-btn"])

                with gr.Column(scale=4):
                    audio_out_t1 = gr.Audio(label="Generated Audio Output", type="filepath")
                    status_out_t1 = gr.Markdown("Ready for generation.")

                    with gr.Accordion("💡 Pro-Tips: How to Get the Strongest Emotion", open=True):
                        gr.Markdown("""
                        1. **Physical + Emotion Pairings (Key Synergy):** Combining physical state tags with emotion produces the most realistic acting: e.g. `[panting] [tired]`, `[laughing] [excited]`, `[clears throat] [angry]`, `[sighing] [emphasis]`.
                        2. **Open-Domain Acting Directives:** Fish Audio S2.1 supports complete natural language directing phrases! Direct the performance like a human voice actor: e.g. `[the calm, measured tone of someone who has done this a thousand times]`, `[dead tired, end of a very long shift]`, `[voice rough from crying, trying to sound normal]`.
                        3. **Place tags before the words:** Tags only affect text that follows them. Placing them at sentence or clause starts produces the strongest shift.
                        4. **Match text with emotion:** The Dual-AR language model fuses semantic meaning with acoustic tokens. Emotionally charged phrasing yields dramatically richer prosody.
                        5. **Use expressive punctuation:** Exclamation marks (`!`), ellipses (`...`), and question marks (`?`) guide the decoder to add natural breath pauses, pitch rises, and vocal tension.
                        6. **Clone expressive voices:** Use Tab 2 to clone a dynamic reference clip for even wider emotional range than the neutral default narrator.
                        """)

            # Tab 1 Event Bindings
            preset_dropdown.change(
                fn=lambda name: PRESET_SCRIPTS.get(name, ""),
                inputs=[preset_dropdown],
                outputs=[text_input_t1]
            )

            tag_buttons = [
                (t_happy, "[happy]"), (t_sad, "[sad]"), (t_angry, "[angry]"), (t_excited, "[excited]"),
                (t_nervous, "[nervous]"), (t_calm, "[calm]"), (t_sarcastic, "[sarcastic]"), (t_fearful, "[fearful]"),
                (t_tired, "[tired]"),
                (t_clearthroat, "[clears throat]"), (t_sighing, "[sighing]"),
                (t_whisper, "[whispering]"), (t_laugh, "[laughing]"), (t_sighs, "[sighs]"), (t_giggles, "[giggles]"),
                (t_crying, "[crying]"), (t_gasping, "[gasping]"), (t_sniffling, "[sniffling]"), (t_panting, "[panting]"),
                (t_softly, "[softly]"), (t_vbreak, "[voice breaking]"), (t_emphasis, "[emphasis]"),
                (t_shouting, "[shouting]"), (t_breathless, "[breathless]"), (t_fast, "[fast paced]"),
                (t_slow, "[slow paced]"),
                (t_measured, "[the calm, measured tone of someone who has done this a thousand times]"),
                (t_deadtired, "[dead tired, end of a very long shift]"),
                (t_cryingmask, "[voice rough from crying, trying to sound normal]")
            ]
            for btn, tag_val in tag_buttons:
                btn.click(fn=insert_emotion_tag, inputs=[text_input_t1, gr.State(tag_val)], outputs=[text_input_t1])

            t1_event = gen_btn_t1.click(
                fn=generate_expressive_tts,
                inputs=[api_key_input, model_select_t1, text_input_t1, voice_dropdown_t1, manual_ref_id_t1, format_t1],
                outputs=[audio_out_t1, status_out_t1]
            )
            stop_btn_t1.click(fn=None, cancels=[t1_event])
            clear_btn_t1.click(
                fn=lambda: (None, "Ready for generation.", "", "", ""),
                outputs=[audio_out_t1, status_out_t1, text_input_t1, voice_dropdown_t1, manual_ref_id_t1]
            )
            refresh_voices_btn.click(
                fn=get_user_voice_choices,
                inputs=[api_key_input, voice_filter_t1],
                outputs=[voice_dropdown_t1]
            )
            voice_filter_t1.change(
                fn=get_user_voice_choices,
                inputs=[api_key_input, voice_filter_t1],
                outputs=[voice_dropdown_t1]
            )
            api_key_input.change(
                fn=get_user_voice_choices,
                inputs=[api_key_input, voice_filter_t1],
                outputs=[voice_dropdown_t1]
            )

        # -----------------------------------------------------
        # TAB 2: Zero-Shot Voice Cloning (Instant)
        # -----------------------------------------------------
        with gr.Tab("🎙️ Zero-Shot Voice Cloning (Instant)"):
            gr.Markdown("Clone any voice instantly from reference audio. 100% free on `s2.1-pro-free` with no model training or transcript required.")
            with gr.Row():
                with gr.Column(scale=5):
                    ref_audio_input = gr.Audio(
                        label="Reference Voice Audio (Upload 10-30s sample or record via Mic)",
                        type="filepath",
                        sources=["upload", "microphone"]
                    )
                    with gr.Row():
                        model_select_t2 = gr.Dropdown(
                            label="Model",
                            choices=["s2.1-pro-free", "s2.1-pro"],
                            value="s2.1-pro-free",
                            scale=3
                        )
                        format_t2 = gr.Radio(
                            label="Format",
                            choices=["mp3", "wav", "opus"],
                            value="mp3",
                            scale=2
                        )
                    target_text_t2 = gr.Textbox(
                        label="Target Text to Synthesize (Cloned Voice)",
                        placeholder="What do you want this cloned voice to say? You can also use [whispering], [laughing], etc.",
                        value="Hello! This voice was cloned on the fly using Fish Audio S2.1 Pro in just seconds.",
                        lines=4
                    )

                    with gr.Row():
                        gen_btn_t2 = gr.Button("🎬 Clone & Generate", variant="primary", size="lg", elem_classes=["action-gen-btn"])
                        stop_btn_t2 = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_classes=["action-stop-btn"])
                        clear_btn_t2 = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_classes=["action-clear-btn"])

                with gr.Column(scale=5):
                    audio_out_t2 = gr.Audio(label="Cloned Voice Output", type="filepath")
                    status_out_t2 = gr.Markdown("Ready for voice cloning.")

                    gr.Markdown("""
                    ### 🎯 Pro-Tips for Zero-Shot Voice Cloning:
                    1. **Clean Input:** Use 10 to 30 seconds of clean speech with minimal background noise or music.
                    2. **Zero-Shot Engine:** The Dual-AR architecture extracts speaker timbre and style without fine-tuning.
                    3. **Emotion in Clones:** You can combine cloned voices with inline tags like `[excited]` or `[whispering]`!
                    """)

            # Tab 2 Event Bindings
            t2_event = gen_btn_t2.click(
                fn=generate_voice_clone,
                inputs=[api_key_input, model_select_t2, ref_audio_input, target_text_t2, format_t2],
                outputs=[audio_out_t2, status_out_t2]
            )
            stop_btn_t2.click(fn=None, cancels=[t2_event])
            clear_btn_t2.click(
                fn=lambda: (None, "Ready for voice cloning.", None, ""),
                outputs=[audio_out_t2, status_out_t2, ref_audio_input, target_text_t2]
            )

        # -----------------------------------------------------
        # TAB 3: Persistent Custom Voice Studio
        # -----------------------------------------------------
        with gr.Tab("📁 Custom Voice Model Studio"):
            gr.Markdown("Create persistent, reusable custom voice models stored on your account, or inspect your existing collection.")
            with gr.Row():
                with gr.Column(scale=5):
                    gr.Markdown("### ➕ Create a Reusable Voice Model")
                    new_voice_audio = gr.Audio(label="Training Sample Audio (10-30s clean audio)", type="filepath")
                    new_voice_title = gr.Textbox(label="Voice Model Title", placeholder="e.g. Studio Narrator Mark")
                    new_voice_desc = gr.Textbox(label="Description (Optional)", placeholder="e.g. Deep, calm male voice for documentaries")
                    new_voice_vis = gr.Dropdown(label="Visibility", choices=["private", "public"], value="private")

                    create_voice_btn = gr.Button("🚀 Create Reusable Voice Model", variant="primary", elem_classes=["action-gen-btn"])
                    create_status_out = gr.Markdown("")

                with gr.Column(scale=5):
                    gr.Markdown("### 📋 Browse Your Voice Models")
                    self_only_chk = gr.Checkbox(label="Show Only My Created Voices", value=True)
                    fetch_voices_btn = gr.Button("🔄 Refresh / List Voices", variant="secondary")
                    voices_table_out = gr.Markdown("Click 'Refresh / List Voices' to view your voice models and copy their IDs.")

            # Tab 3 Event Bindings
            def create_voice_and_update(api_key, audio, title, desc, vis, filter_m):
                status_msg = create_persistent_voice(api_key, audio, title, desc, vis)
                dropdown_upd = get_user_voice_choices(api_key, filter_m)
                return status_msg, dropdown_upd

            create_voice_btn.click(
                fn=create_voice_and_update,
                inputs=[api_key_input, new_voice_audio, new_voice_title, new_voice_desc, new_voice_vis, voice_filter_t1],
                outputs=[create_status_out, voice_dropdown_t1]
            )
            fetch_voices_btn.click(
                fn=list_user_voices,
                inputs=[api_key_input, self_only_chk],
                outputs=[voices_table_out]
            )

    # Load account & system voices on startup
    demo.load(fn=get_user_voice_choices, inputs=[api_key_input, voice_filter_t1], outputs=[voice_dropdown_t1])

    # Modern Footer
    gr.HTML("""
    <div class="modern-footer">
      <p style="text-align: center; margin: 0 0 6px 0; font-size: 1.05em; font-weight: 600;">
        🐟 Fish Audio S2.1 Pro Studio • Built with ❤️ by <strong>TushKosh 2.0</strong>
      </p>
      <p style="text-align: center; margin: 0; color: #94a3b8; font-size: 0.9em;">
        Colab Edition (CPU Friendly & 100% Free) • Powered by Gradio 5+ & Fish Audio Foundation API.
      </p>
    </div>
    """)

# Launch the Gradio application
if __name__ == "__main__":
    demo.queue().launch(share=True, debug=True)

<hr style="border:none;border-top:1px solid #C9A84C;margin:30px 0 20px;" /><div align="center" style="background:linear-gradient(135deg,#0A0A0A 0%,#111111 60%,#0D0D0D 100%);border:1px solid #C9A84C;border-radius:16px;padding:28px 24px 24px;margin:0 0 10px;box-shadow:0 0 30px rgba(201,168,76,0.12),0 4px 24px rgba(0,0,0,0.5);font-family:Arial,sans-serif;text-align:center;"><div style="width:60px;height:2px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);margin:0 auto 16px;border-radius:2px;"></div><h2 align="center" style="color:#FFFFFF;margin:0 0 6px;font-size:1.5em;letter-spacing:2px;text-align:center;">TushKosh 2.0</h2><p align="center" style="color:#C9A84C;margin:0 0 20px;font-weight:700;letter-spacing:3px;font-size:12px;text-transform:uppercase;text-align:center;">Learn &bull; Build &bull; Create</p><div align="center" style="text-align:center;margin:0 0 18px;"><a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" style="display:inline-block;background:linear-gradient(135deg,#C9A84C 0%,#FFD700 50%,#C9A84C 100%);color:#000000;font-weight:800;font-size:13px;padding:11px 28px;border-radius:50px;text-decoration:none;letter-spacing:1px;box-shadow:0 4px 18px rgba(201,168,76,0.45);">&#9654; Subscribe on YouTube</a></div><p align="center" style="color:#888888;font-size:12px;margin:0 0 6px;text-align:center;">Original notebook by <strong style="color:#AAAAAA;">AIQUEST Academy</strong></p><p align="center" style="color:#666666;font-size:11px;margin:0;text-align:center;">TushKosh 2.0 branding added &middot; Original code and third-party components attributed to their respective creators.</p><div style="width:60px;height:2px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);margin:16px auto 0;border-radius:2px;"></div></div>